# Twitter Sentiment Analysis

In [1]:
# importing dependencies
import numpy as np
import pandas as pd
import re
import nltk
import pickle
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score

nltk.download('stopwords')
print('All imports done!')

All imports done!


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/priyannshuppal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Load the dataset — single clean load with column names
column_names = ['target', 'ids', 'date', 'flag', 'user', 'text']
twitter_data = pd.read_csv(
    'training.1600000.processed.noemoticon.csv',
    names=column_names,
    header=None,
    encoding='ISO-8859-1'
)
print('Shape:', twitter_data.shape)
display(twitter_data.head())

Shape: (1600000, 6)


,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [3]:
# Check missing values
print(twitter_data.isnull().sum())

target    0
ids       0
date      0
flag      0
user      0
text      0
dtype: int64


In [4]:
# Check target distribution (0 = negative, 4 = positive)
print(twitter_data['target'].value_counts())

target
0    800000
4    800000
Name: count, dtype: int64


In [5]:
# Convert target: 4 -> 1  (0=negative, 1=positive)
twitter_data['target'] = twitter_data['target'].replace(4, 1)
print(twitter_data['target'].value_counts())

target
0    800000
1    800000
Name: count, dtype: int64


In [6]:
# Stemming function
ps = PorterStemmer()

def stemming(content):
    stemmed = re.sub('[^a-zA-Z]', ' ', content)
    stemmed = stemmed.lower().split()
    stemmed = [ps.stem(w) for w in stemmed if w not in stopwords.words('english')]
    return ' '.join(stemmed)

In [7]:
# Apply stemming (this takes ~5-10 mins on 1.6M rows)
print('Applying stemming... please wait')
twitter_data['stemmed_content'] = twitter_data['text'].apply(stemming)
print('Done!')
display(twitter_data.head())

Applying stemming... please wait
Done!


,target,ids,date,flag,user,text,stemmed_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


In [12]:
# Separate features and labels
X = twitter_data['stemmed_content'].values
Y = twitter_data['target'].values
print('X shape:', X.shape)
print('Y shape:', Y.shape)

X shape: (1600000,)
Y shape: (1600000,)


In [9]:
# Train/test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=2
)
print('Train:', X_train.shape, '  Test:', X_test.shape)

Train: (1280000,)   Test: (320000,)


In [10]:
# TF-IDF Vectorization
vectorizer = TfidfVectorizer()
vectorizer.fit(X_train)
X_train = vectorizer.transform(X_train)
X_test  = vectorizer.transform(X_test)
print('Vectorization done!')

Vectorization done!


In [13]:
# Train Linear SVM model (better accuracy than Logistic Regression)
model = CalibratedClassifierCV(LinearSVC(max_iter=2000))
model.fit(X_train, Y_train)
print('Model trained!')

/Users/priyannshuppal/sentiment_app/venv/lib/python3.13/site-packages/sklearn/calibration.py:837: RuntimeWarning: divide by zero encountered in matmul
  grad = np.asarray([-g @ F, -g.sum()], dtype=np.float64)
/Users/priyannshuppal/sentiment_app/venv/lib/python3.13/site-packages/sklearn/calibration.py:837: RuntimeWarning: overflow encountered in matmul
  grad = np.asarray([-g @ F, -g.sum()], dtype=np.float64)
/Users/priyannshuppal/sentiment_app/venv/lib/python3.13/site-packages/sklearn/calibration.py:837: RuntimeWarning: invalid value encountered in matmul
  grad = np.asarray([-g @ F, -g.sum()], dtype=np.float64)
/Users/priyannshuppal/sentiment_app/venv/lib/python3.13/site-packages/sklearn/calibration.py:837: RuntimeWarning: divide by zero encountered in matmul
  grad = np.asarray([-g @ F, -g.sum()], dtype=np.float64)
/Users/priyannshuppal/sentiment_app/venv/lib/python3.13/site-packages/sklearn/calibration.py:837: RuntimeWarning: overflow encountered in matmul
  grad = np.asarray([-g @ 

Model trained!


/Users/priyannshuppal/sentiment_app/venv/lib/python3.13/site-packages/sklearn/calibration.py:837: RuntimeWarning: divide by zero encountered in matmul
  grad = np.asarray([-g @ F, -g.sum()], dtype=np.float64)
/Users/priyannshuppal/sentiment_app/venv/lib/python3.13/site-packages/sklearn/calibration.py:837: RuntimeWarning: overflow encountered in matmul
  grad = np.asarray([-g @ F, -g.sum()], dtype=np.float64)
/Users/priyannshuppal/sentiment_app/venv/lib/python3.13/site-packages/sklearn/calibration.py:837: RuntimeWarning: invalid value encountered in matmul
  grad = np.asarray([-g @ F, -g.sum()], dtype=np.float64)


In [14]:
# Evaluate accuracy
train_acc = accuracy_score(model.predict(X_train), Y_train)
test_acc  = accuracy_score(model.predict(X_test),  Y_test)
print(f'Training accuracy: {train_acc:.4f}')
print(f'Test accuracy:     {test_acc:.4f}')

Training accuracy: 0.8575
Test accuracy:     0.7717


In [15]:
# Save model AND vectorizer
pickle.dump(model,      open('trained_model.sav', 'wb'))
pickle.dump(vectorizer, open('vectorizer.sav',    'wb'))
print('Both files saved!')

Both files saved!


In [16]:
# Quick sanity test
loaded_model      = pickle.load(open('trained_model.sav', 'rb'))
loaded_vectorizer = pickle.load(open('vectorizer.sav',    'rb'))

def preprocess(text):
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower().split()
    text = [ps.stem(w) for w in text if w not in stopwords.words('english')]
    return ' '.join(text)

tests = [
    'I love this beautiful day!',
    'This is the worst experience ever',
    'Feeling amazing today!'
]

for t in tests:
    vec  = loaded_vectorizer.transform([preprocess(t)])
    pred = loaded_model.predict(vec)[0]
    print(f"'{t}' → {'Positive' if pred==1 else 'Negative'}")

'I love this beautiful day!' → Positive
'This is the worst experience ever' → Negative
'Feeling amazing today!' → Positive
